# Create Databricks Vector Search Endpoint and Index

This notebook provisions (or reuses) a Vector Search endpoint and creates a managed Delta Sync index for policy chunks in `iitb.bharat_bricks.policy_chunks_silver`.

In [ ]:
%pip install databricks-vectorsearch

In [ ]:
from datetime import timedelta

from databricks.sdk.errors import NotFound, ResourceDoesNotExist
from databricks.vector_search.client import VectorSearchClient

CATALOG = "iitb"
SCHEMA = "bharat_bricks"
ENDPOINT_NAME = "bhujal_mitra_endpoint"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.bhujal_mitra_policy_index"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.policy_chunks_silver"
PRIMARY_KEY_COL = "chunk_id"
TEXT_COL = "chunk_text"
EMBEDDING_MODEL = "databricks-bge-large-en"

vsc = VectorSearchClient()
print("VectorSearchClient initialized.")

source_df = spark.table(SOURCE_TABLE).select(PRIMARY_KEY_COL, TEXT_COL, "source_name")
source_count = source_df.count()
print(f"Source table: {SOURCE_TABLE}")
print(f"Source rows: {source_count}")

if source_count == 0:
    raise ValueError(f"Source table {SOURCE_TABLE} is empty. Run policy chunking first.")

spark.sql(
    f"ALTER TABLE {SOURCE_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
 )
print("Enabled Delta Change Data Feed on source table.")
display(spark.sql(f"SHOW TBLPROPERTIES {SOURCE_TABLE} ('delta.enableChangeDataFeed')"))

display(source_df.limit(10))

In [ ]:
endpoint_exists = True

try:
    endpoint_info = vsc.get_endpoint(ENDPOINT_NAME)
    print(f"Endpoint already exists: {ENDPOINT_NAME}")
except Exception as exc:
    msg = str(exc).lower()
    if "not found" in msg or "does not exist" in msg:
        endpoint_exists = False
        print(f"Endpoint not found. Creating endpoint: {ENDPOINT_NAME}")
    else:
        raise

if not endpoint_exists:
    vsc.create_endpoint_and_wait(
        name=ENDPOINT_NAME,
        endpoint_type="STANDARD",
        verbose=True,
        timeout=timedelta(minutes=90),
    )

print(f"Waiting for endpoint to be ready: {ENDPOINT_NAME}")
vsc.wait_for_endpoint(
    name=ENDPOINT_NAME,
    verbose=True,
    timeout=timedelta(minutes=90),
)

endpoint_info = vsc.get_endpoint(ENDPOINT_NAME)
print("Endpoint status:", endpoint_info.get("endpoint_status", endpoint_info.get("status", {})))

In [ ]:
index_exists = True

try:
    index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
    print(f"Index already exists: {INDEX_NAME}")
except Exception as exc:
    msg = str(exc).lower()
    if "not found" in msg or "does not exist" in msg:
        index_exists = False
        print(f"Index not found. Creating managed Delta Sync index: {INDEX_NAME}")
    else:
        raise

if not index_exists:
    vsc.create_delta_sync_index(
        endpoint_name=ENDPOINT_NAME,
        index_name=INDEX_NAME,
        primary_key=PRIMARY_KEY_COL,
        source_table_name=SOURCE_TABLE,
        pipeline_type="TRIGGERED",
        embedding_source_column=TEXT_COL,
        embedding_model_endpoint_name=EMBEDDING_MODEL,
    )
    index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)

index_info = index.describe()
print("Endpoint:", ENDPOINT_NAME)
print("Index:", INDEX_NAME)
print("Source table:", SOURCE_TABLE)
print("Primary key column:", PRIMARY_KEY_COL)
print("Embedding source column:", TEXT_COL)
print("Embedding model:", EMBEDDING_MODEL)
print("Index status:", index_info.get("status", {}))